# 09 V2 Year-Over-Year Comparison

This notebook splits the expanded two-year Online Retail II dataset into two trading periods and compares commercial retention patterns year over year.

Periods:

- Prior year: 2009-12-01 to 2010-11-30
- Current year: 2010-12-01 to 2011-12-09

The current-year period aligns with the original one-year project baseline.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

PROCESSED_DIR = Path("../data/processed/v2_online_retail_ii")
TRANSACTIONS_PATH = PROCESSED_DIR / "clean_transactions.parquet"

if not TRANSACTIONS_PATH.exists():
    raise FileNotFoundError(
        f"Expected V2 processed transaction layer not found: {TRANSACTIONS_PATH}"
    )

In [ ]:
transactions = pd.read_parquet(TRANSACTIONS_PATH)
transactions["InvoiceDate"] = pd.to_datetime(transactions["InvoiceDate"])

periods = {
    "prior_year_2009_12_to_2010_11": ("2009-12-01", "2010-11-30 23:59:59"),
    "current_year_2010_12_to_2011_12": ("2010-12-01", "2011-12-09 23:59:59"),
}

print(f"V2 rows loaded: {len(transactions):,}")
print(f"Date range: {transactions['InvoiceDate'].min()} to {transactions['InvoiceDate'].max()}")

In [ ]:
def period_slice(start, end):
    return transactions[
        (transactions["InvoiceDate"] >= pd.Timestamp(start))
        & (transactions["InvoiceDate"] <= pd.Timestamp(end))
    ].copy()


def build_orders(period_transactions):
    valid = period_transactions[period_transactions["is_valid_order_line"]].copy()
    orders = (
        valid.groupby(["CustomerID", "InvoiceNo"], as_index=False)
        .agg(
            order_date=("InvoiceDate", "min"),
            order_revenue=("analytical_revenue", "sum"),
        )
    )
    orders = orders[orders["order_revenue"] > 0].sort_values(
        ["CustomerID", "order_date", "InvoiceNo"]
    )
    orders["order_rank"] = orders.groupby("CustomerID").cumcount() + 1
    return orders


def period_metrics(period_name, start, end):
    period_transactions = period_slice(start, end)
    orders = build_orders(period_transactions)
    customers = (
        orders.groupby("CustomerID")
        .agg(
            total_orders=("InvoiceNo", "nunique"),
            revenue=("order_revenue", "sum"),
            first_purchase=("order_date", "min"),
            last_purchase=("order_date", "max"),
        )
        .reset_index()
    )

    first_orders = orders[orders["order_rank"].eq(1)][["CustomerID", "order_date"]].rename(
        columns={"order_date": "first_order_date"}
    )
    second_orders = orders[orders["order_rank"].eq(2)][["CustomerID", "order_date"]].rename(
        columns={"order_date": "second_order_date"}
    )
    second_days = first_orders.merge(second_orders, on="CustomerID")

    first_invoice = orders[orders["order_rank"].eq(1)][["CustomerID", "InvoiceNo"]].assign(
        is_first_order=True
    )
    lifecycle = orders.merge(first_invoice, on=["CustomerID", "InvoiceNo"], how="left")

    total_revenue = customers["revenue"].sum()
    repeat_revenue = lifecycle.loc[lifecycle["is_first_order"].ne(True), "order_revenue"].sum()
    customers_sorted = customers.sort_values(
        ["revenue", "CustomerID"], ascending=[False, True]
    ).reset_index(drop=True)
    top_n = int(np.floor(1 + (len(customers_sorted) - 1) * 0.1))

    gross_purchase_revenue = period_transactions.loc[
        period_transactions["is_positive_purchase"], "line_revenue_gross"
    ].sum()
    refund_revenue = period_transactions.loc[
        period_transactions["is_refund_or_return"], "line_revenue_gross"
    ].sum()

    return {
        "period": period_name,
        "start_date": pd.Timestamp(start).date().isoformat(),
        "end_date": pd.Timestamp(end).date().isoformat(),
        "total_rows": len(period_transactions),
        "duplicate_rows": int(period_transactions.duplicated().sum()),
        "missing_customer_pct": period_transactions["CustomerID"].isna().mean(),
        "gross_purchase_revenue": gross_purchase_revenue,
        "net_transactional_revenue": gross_purchase_revenue + refund_revenue,
        "identifiable_customer_revenue": total_revenue,
        "identifiable_revenue_pct": total_revenue / gross_purchase_revenue,
        "valid_customers": len(customers),
        "valid_orders": orders["InvoiceNo"].nunique(),
        "second_purchase_conversion": (customers["total_orders"] > 1).mean(),
        "median_days_to_second_purchase": float(
            (second_days["second_order_date"] - second_days["first_order_date"]).dt.days.median()
        ),
        "repeat_order_revenue_share": repeat_revenue / total_revenue,
        "repeat_customer_revenue_share": customers.loc[
            customers["total_orders"] > 1, "revenue"
        ].sum() / total_revenue,
        "top_10_revenue_share": customers_sorted.head(top_n)["revenue"].sum() / total_revenue,
    }

In [ ]:
period_summary = pd.DataFrame(
    [period_metrics(period_name, start, end) for period_name, (start, end) in periods.items()]
)
period_summary

In [ ]:
period_frames = {period_name: period_slice(start, end) for period_name, (start, end) in periods.items()}
prior = period_frames["prior_year_2009_12_to_2010_11"]
current = period_frames["current_year_2010_12_to_2011_12"]

prior_customers = set(prior["CustomerID"].dropna().astype(int))
current_customers = set(current["CustomerID"].dropna().astype(int))
prior_stock = set(prior["StockCode"].astype(str))
current_stock = set(current["StockCode"].astype(str))
prior_invoices = set(prior["InvoiceNo"].astype(str))
current_invoices = set(current["InvoiceNo"].astype(str))

period_overlap = pd.DataFrame(
    {
        "metric": [
            "customer_overlap_count",
            "prior_customer_overlap_pct",
            "current_customer_overlap_pct",
            "stock_overlap_count",
            "prior_stock_overlap_pct",
            "current_stock_overlap_pct",
            "invoice_overlap_count",
        ],
        "value": [
            len(prior_customers & current_customers),
            len(prior_customers & current_customers) / len(prior_customers),
            len(prior_customers & current_customers) / len(current_customers),
            len(prior_stock & current_stock),
            len(prior_stock & current_stock) / len(prior_stock),
            len(prior_stock & current_stock) / len(current_stock),
            len(prior_invoices & current_invoices),
        ],
    }
)
period_overlap

In [ ]:
period_summary.to_csv(PROCESSED_DIR / "v2_year_over_year_period_summary.csv", index=False)
period_overlap.to_csv(PROCESSED_DIR / "v2_year_over_year_overlap.csv", index=False)

for output_name in ["v2_year_over_year_period_summary.csv", "v2_year_over_year_overlap.csv"]:
    output_path = PROCESSED_DIR / output_name
    if not output_path.exists():
        raise FileNotFoundError(f"Expected output was not created: {output_path}")
    print(f"Saved {output_path}")

## Interpretation

The two periods tell a stable commercial story. Repeat revenue remains high, customer concentration remains material, and second purchase timing stays close to 50 days. The current year has slightly lower repeat conversion and repeat-order revenue share, while customer concentration increases.